In [ ]:
# [목적] AI 답변을 정해 둔 항목별 데이터로 받기 위한 도구와 환경을 준비합니다.
# ResponseSchema는 받을 항목을 정하고, StructuredOutputParser는 답변을 그 형식으로 읽습니다.
# [중요] 형식이 필요한 이유는 AI 답변을 프로그램이 자동으로 활용하기 위해서입니다.
# load_dotenv는 API 키를 읽고 logging은 모델 실행 기록을 남깁니다.
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("Chapter6-OutputParser")

In [ ]:
# [목적] AI 답변에서 꼭 받을 정보의 이름과 설명을 정합니다.
# answer는 답변, source는 답변의 근거가 될 출처입니다.
# [중요] description을 구체적으로 쓸수록 AI가 각 칸을 더 정확히 채웁니다.
response_schemas = [
    ResponseSchema(name="answer", description="사용자의 질문에 대한 답변"),
    ResponseSchema(
        name="source",
        description="사용자의 질문에 답하기 위해 사용된 '출처', '웹사이트 주소'이어야 합니다.",
    ),
]

In [ ]:
# [목적] 위에서 정한 answer·source 형식에 맞춰 답변을 읽을 파서를 만듭니다.
# 나중에 AI의 텍스트 답변을 Python 딕셔너리 형태로 바꿔 줍니다.
# [중요] 파서가 없으면 answer와 source를 코드에서 안정적으로 꺼내기 어렵습니다.
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [ ]:
# [목적] AI에게 전달할 출력 형식 안내문을 확인합니다.
# 이 안내문 덕분에 AI가 answer와 source를 빠뜨리지 않게 됩니다.
# 출력 형식이 어떻게 생겼는지 눈으로 확인하는 학습용 셀입니다.
print(output_parser.get_format_instructions())

In [5]:
# [목적] 질문과 출력 형식 안내를 합친 프롬프트를 만듭니다.
# {question}만 실행할 때 바뀌고, 형식 안내는 미리 고정해 둡니다.
# [중요] partial_variables는 반복되는 안내문을 매번 입력하지 않게 합니다.
format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    template="answer the users question as best as possible.\n{format_instructions}\n{question}",
    input_variables=["question"],
    partial_variables={"format_instructions": format_instructions},
)

In [6]:
# [목적] 프롬프트 → AI 모델 → 파서를 하나의 실행 흐름으로 연결합니다.
# temperature=0은 답변의 무작위성을 낮춰 결과를 일정하게 만듭니다.
# [중요] | 기호는 앞 단계의 결과를 다음 단계로 전달한다는 뜻입니다.
model = ChatOpenAI(temperature=0)
chain = prompt | model | output_parser

In [7]:
# [목적] 질문을 실행하고 answer와 source가 담긴 구조화된 결과를 받습니다.
# 이런 결과는 화면 표시나 데이터베이스 저장에 바로 쓰기 좋습니다.
# invoke는 작업을 한 번 실행하고 최종 결과를 받는 메서드입니다.
chain.invoke({"question": "대한민국의 수도는 어디인가요?"})

{'answer': '서울', 'source': 'https://ko.wikipedia.org/wiki/%EC%84%9C%EC%9A%B8'}